In [2]:
from farmdar.secrets import set_aws_keys
from farmdar.auth import refresh_token
import s3fs
from farmdar.data import insert_df
import dask_geopandas as dgpd
import dask
from dask.distributed import Client
import re
import geopandas as gpd
from collections import defaultdict
import pandas as pd
from datetime import datetime
set_aws_keys()

In [3]:
def get_client_point_stats(bucket_name='centralized-data-storage'):
    """
    Calculate total survey point files and total point features per client
    + add grand total row
    """
    fs = s3fs.S3FileSystem()
    
    # To store stats
    stats = []
    grand_total_points = 0
    grand_total_files = 0
    grand_total_geojson = 0
    
    # Find all clients in bucket
    try:
        bucket_contents = fs.ls(bucket_name)
        clients = [item.split('/')[-1] for item in bucket_contents if fs.isdir(item)]
    except Exception as e:
        print(f"Error listing clients: {e}")
        return pd.DataFrame()

    for client in clients:
        client_refined_path = f"{bucket_name}/{client}/h_survey_points/refined"
        
        if not fs.exists(client_refined_path):
            continue
        
        # Recursive glob to catch nested folders
        all_items = fs.glob(f"{client_refined_path}/**")
        files = [f for f in all_items if fs.isfile(f)]
        geojson_files = [f for f in files if f.lower().endswith(('.geojson', '.json'))]
        
        total_points = 0
        
        for file_path in geojson_files:
            try:
                # Load file content
                with fs.open(file_path, 'rb') as fobj:
                    raw = json.load(fobj)
                
                if "features" not in raw:
                    continue
                
                # Count features that are Points
                for feat in raw["features"]:
                    geom = feat.get("geometry", {})
                    if geom and geom.get("type") == "Point":
                        total_points += 1
            
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                continue
        
        stats.append({
            "client_name": client,
            "total_files": len(files),
            "geojson_files": len(geojson_files),
            "total_point_features": total_points
        })
        
        grand_total_files += len(files)
        grand_total_geojson += len(geojson_files)
        grand_total_points += total_points
    
    # ✅ convert to dataframe
    df_points = pd.DataFrame(stats).sort_values("total_point_features", ascending=False).reset_index(drop=True)
    
    # ✅ add TOTAL row
    df_points.loc[len(df_points.index)] = {
        "client_name": "TOTAL",
        "total_files": grand_total_files,
        "geojson_files": grand_total_geojson,
        "total_point_features": grand_total_points
    }
    
    return df_points


In [4]:
df_client_points = get_client_point_stats()
df_client_points


,client_name,total_files,geojson_files,total_point_features
0,Corteva,7,7,38717
1,Al-Moiz,6,6,5534
2,FSML,4,4,3200
3,Sheikhoo,3,3,3050
4,RNSM,2,2,2155
5,Adam,3,3,2015
6,FFC,2,2,1872
7,Faran,3,3,1853
8,Centrigo,4,4,1422
9,HSM,3,3,1366
